# NASA–Air Force–DOE commercialization outcomes

**Status:** exploratory — non-citable  
**Research questions:** B2/B3, F1/F3  
**Decision this informs:** how three agency portfolios differ in observed federal-contract, private-capital, and acquisition pathways  
**Data cutoff:** 2024-12-31

This notebook is a diagnostic companion. The canonical calculations live in `scripts/data/three_agency_commercialization_outcomes.py`.

## Data contract

- **Population:** NASA, Air Force, and DOE firms with a first SBIR/STTR Phase II award from 2009 onward.
- **Grain:** firm × funding agency; firms may appear in more than one agency cohort.
- **Anchor:** first Phase II award date from that agency.
- **Outcomes:** observed prime-contract, Form D, and public-filing M&A signals within fixed horizons.
- **Missingness:** no observed signal is not evidence that commercialization did not occur.
- **Inference:** descriptive only; no causal or composite-ranking claim.

In [ ]:
import os
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'sbir_etl').exists():
            return candidate
    raise RuntimeError('Run this notebook from inside the sbir-analytics checkout')


REPO_ROOT = find_repo_root()
REPORT_DIR = Path(os.environ.get(
    'SBIR_THREE_AGENCY_REPORT_DIR',
    REPO_ROOT / 'data/reports/three_agency_commercialization',
))
OUTCOMES = REPORT_DIR / 'agency_horizon_outcomes.csv'
OVERLAP = REPORT_DIR / 'channel_overlap.csv'
if not OUTCOMES.exists():
    raise FileNotFoundError('Run scripts/data/three_agency_commercialization_outcomes.py first')
outcomes = pd.read_csv(OUTCOMES)
overlap = pd.read_csv(OVERLAP)


## Five-year scorecard

In [ ]:
primary = outcomes[(outcomes.horizon_years == 5) & ((outcomes.channel == 'federal_contract') | (outcomes.confidence_filter == 'high'))]
primary[['agency', 'channel', 'eligible_firms', 'firms_with_signal', 'rate', 'rate_ci_low', 'rate_ci_high', 'dollars_per_phase_ii_dollar', 'median_latency_years']]

## Horizon and confidence sensitivity

In [ ]:
outcomes.pivot_table(index=['agency', 'horizon_years'], columns=['channel', 'confidence_filter'], values='rate')

## Five-year channel overlap

In [ ]:
overlap.sort_values(['agency', 'firms'], ascending=[True, False])

## Interpretation checklist

- Keep incidence and dollar intensity separate.
- Compare high-only SEC results with high+medium sensitivity before interpreting agency differences.
- Treat Form D and M&A as public-disclosure lower bounds.
- Check the generated manifest's amendment and linkage audits before quoting a value internally.
- Do not attribute an Air Force time break to AFWERX without a separate causal design.